# 17 — ReAct RAG

> **Tier 3 | Query Handling**

## The Problem

CoT RAG (nb 16) plans all reasoning steps upfront. But what if you don't know
in advance how many steps are needed, or which tool to call next?

```
Q: "How does the relationship between ocean heat content and hurricane intensity
    relate to recent trends in Atlantic storms?"

After reading about ocean heat → you discover you need a specific lookup on
sea surface temperature thresholds.
After that → you need to search for Atlantic storm frequency data.
Only then can you connect the two and answer.

You couldn't have planned all three searches upfront.
```

## ReAct Solution

**ReAct** (Reason + Act) interleaves three operations in a loop until the model
decides it has enough information to answer:

| Turn | What happens |
|------|--------------|
| **Thought** | LLM reasons about what it knows and what it still needs |
| **Action** | LLM picks a tool: `search(query)`, `lookup(term)`, or `finish(answer)` |
| **Observation** | Tool result is appended to the scratchpad |

The model sees the **full scratchpad** on every turn — every thought, action,
and observation so far — so each decision is informed by all prior results.

## Tools Available

| Tool | Signature | Purpose |
|------|-----------|--------|
| `search` | `search("query")` | Semantic vector search — broad retrieval |
| `lookup` | `lookup("term")` | Targeted term lookup — precise retrieval |
| `finish` | `finish("answer")` | Terminate loop with final answer |

## Flow Diagram

```mermaid
flowchart TD
    Q(["❓ Question"])
    Q --> T1["💭 Thought\nWhat do I know? What do I need?"]

    subgraph LOOP ["🔁  Thought / Action / Observation Loop"]
        direction TB
        A1["⚡ Action\nsearch / lookup / finish"]
        O1["👁 Observation\ntool result appended to scratchpad"]
        T2["💭 Thought\nre-evaluate with new evidence"]
        A1 --> O1 --> T2 --> A1
    end

    T1 --> LOOP
    LOOP -->|"finish(answer) called"| ANS(["✅ Final answer"])

    style LOOP fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style ANS  fill:#dcfce7,stroke:#16a34a,color:#14532d
```

## Step 1 — Install Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3","qdrant-client","opensearch-py","requests-aws4auth",
            "strands-agents","pypdf","python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")

## Step 2 — Imports & Configuration

In [ ]:
import os, json, time, uuid, re
from typing import List, Dict, Optional, Tuple

import boto3, pypdf
from dotenv import load_dotenv
from strands import Agent
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)

AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

COLLECTION_NAME = "react_rag_17"
EMBEDDING_DIM   = 1024
TOP_K           = 3     # passages per tool call (kept small — model calls multiple times)
MAX_ITERATIONS  = 8     # hard cap on Thought/Action/Observation cycles
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
PDF_PATH        = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection     : {COLLECTION_NAME}")
print(f"PDF exists     : {os.path.exists(PDF_PATH)}")
print(f"AWS region     : {AWS_REGION}")
print(f"Key ID         : {os.getenv('AWS_ACCESS_KEY_ID','NOT SET')[:12]}...")
print(f"Max iterations : {MAX_ITERATIONS}")

## Step 3 — Vector Store

In [ ]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name = collection_name
        self._backend = None
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}')
                return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}')
                return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name); exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
                print(f'Created "{self.name}" (dim={dim})')
            else:
                print(f'"{self.name}" already exists — skipping recreate')

    def upsert(self, docs: List[Dict]) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                               payload={'text': d['text'], 'metadata': d.get('metadata', {})})
                   for d in docs]
            self._qdrant.upsert(collection_name=self.name, points=pts)
            return len(pts)

    def search(self, qvec: List[float], top_k: int = 5) -> List[Dict]:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            resp = self._qdrant.query_points(
                collection_name=self.name, query=qvec, limit=top_k,
                with_payload=True)
            return [{'text': p.payload.get('text', ''),
                     'metadata': p.payload.get('metadata', {}),
                     'score': p.score, 'id': str(p.id)}
                    for p in resp.points]
        return []

    def count(self) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            return self._qdrant.get_collection(self.name).points_count or 0
        return 0

    def delete_collection(self):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            self._qdrant.delete_collection(self.name)
        print(f'Deleted "{self.name}"')

print("VectorStore defined.")

## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

def llm(prompt: str) -> str:
    return str(Agent(model=_model, system_prompt="You are a helpful assistant.")(prompt))

test_emb = embed_text("ReAct RAG test")
print(f"Embedding OK — dim={len(test_emb)}")
print("Bedrock LLM ready.")

## Step 5 — Connect & Index climate.pdf

In [ ]:
def recursive_split(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    separators = ["\n\n", "\n", ". ", " ", ""]
    def _split(text, seps):
        sep, new_seps = '', []
        for i, s in enumerate(seps):
            if s == '' or s in text:
                sep, new_seps = s, seps[i+1:]; break
        parts = text.split(sep) if sep != '' else list(text)
        good = []
        for part in parts:
            if len(part) <= chunk_size: good.append(part)
            elif new_seps: good.extend(_split(part, new_seps))
            else:
                for k in range(0, len(part), chunk_size - chunk_overlap):
                    good.append(part[k:k+chunk_size])
        chunks, cur_pieces, cur_len = [], [], 0
        for piece in good:
            p = piece.strip()
            if not p: continue
            addition = len(sep) + len(p) if cur_pieces else len(p)
            if cur_len + addition <= chunk_size:
                cur_pieces.append(p); cur_len += addition
            else:
                if cur_pieces: chunks.append(sep.join(cur_pieces))
                ov, ovl = [], 0
                for prev in reversed(cur_pieces):
                    if ovl + len(prev) + len(sep) <= chunk_overlap:
                        ov.insert(0, prev); ovl += len(prev) + len(sep)
                    else: break
                cur_pieces = ov + [p]
                cur_len = sum(len(x) + len(sep) for x in cur_pieces)
        if cur_pieces: chunks.append(sep.join(cur_pieces))
        return [c for c in chunks if c.strip()]
    return _split(text, separators)

vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL,
    qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL,
    region=AWS_REGION
)
print(f"Backend: {vs._backend}")
vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

reader    = pypdf.PdfReader(PDF_PATH)
full_text = ''
page_map  = []
for pg_idx, page in enumerate(reader.pages):
    pg_text = (page.extract_text() or '') + '\n\n'
    page_map.extend([pg_idx + 1] * len(pg_text))
    full_text += pg_text

chunks = recursive_split(full_text, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"PDF pages: {len(reader.pages)}  |  Chunks: {len(chunks)}")

embs = embed_batch(chunks, label='[index]')
docs = []
for i, (chunk, emb) in enumerate(zip(chunks, embs)):
    char_offset = full_text.find(chunk[:40])
    page_num = page_map[min(char_offset, len(page_map)-1)] if char_offset >= 0 else 1
    docs.append({
        'text'     : chunk,
        'embedding': emb,
        'metadata' : {'chunk_idx': i, 'page_num': page_num, 'char_count': len(chunk)}
    })
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors into '{COLLECTION_NAME}'")

## Step 6 — Tool Definitions

Three tools the ReAct agent can call:

- **`search(query)`** — semantic vector search; good for broad topic retrieval  
- **`lookup(term)`** — targeted search; re-embeds the raw term with no elaboration (precise)  
- **`finish(answer)`** — signals the loop to stop and returns the answer

Both search tools return a compact passage block with page citations so the
LLM can directly reference them in the next Thought.

In [ ]:
def tool_search(query: str, top_k: int = TOP_K) -> str:
    hits = vs.search(embed_text(query), top_k=top_k)
    if not hits:
        return "No results found."
    parts = []
    for h in hits:
        pg = h['metadata'].get('page_num', '?')
        parts.append(f"[p.{pg}] {h['text'][:300]}")
    return '\n---\n'.join(parts)

def tool_lookup(term: str, top_k: int = TOP_K) -> str:
    # Same as search but the query is the raw term — no elaboration
    hits = vs.search(embed_text(term), top_k=top_k)
    if not hits:
        return f"No passages found for '{term}'."
    parts = []
    for h in hits:
        pg = h['metadata'].get('page_num', '?')
        parts.append(f"[p.{pg}] {h['text'][:300]}")
    return '\n---\n'.join(parts)

TOOL_REGISTRY = {
    'search': tool_search,
    'lookup': tool_lookup,
}

# Quick sanity check
print("search('Arctic sea ice') →")
print(tool_search('Arctic sea ice')[:200], '...')

## Step 7 — Action Parser

The LLM outputs one action per turn in the format:

```
Action: search("query string")
Action: lookup("term")
Action: finish("the final answer...")
```

The parser extracts `(tool_name, argument)` from a raw LLM response line.

In [ ]:
# Matches: Action: tool_name("arg") or Action: tool_name('arg')
_ACTION_RE = re.compile(
    r'Action:\s*(\w+)\(\s*["\']?(.*?)["\']?\s*\)',
    re.DOTALL | re.IGNORECASE
)

def parse_action(text: str) -> Optional[Tuple[str, str]]:
    m = _ACTION_RE.search(text)
    if not m:
        return None
    tool = m.group(1).lower().strip()
    arg  = m.group(2).strip()
    return (tool, arg)

# Tests
assert parse_action('Action: search("Arctic feedback loops")') == ('search', 'Arctic feedback loops')
assert parse_action("Action: lookup('permafrost')")            == ('lookup', 'permafrost')
assert parse_action('Action: finish("The answer is X")')       == ('finish', 'The answer is X')
assert parse_action('No action here')                          is None
print("Action parser OK.")

## Step 8 — ReAct Loop

The system prompt teaches the LLM the Thought / Action / Observation format.
On each iteration we:
1. Feed the full scratchpad to the LLM → get `Thought: ... Action: tool(arg)`
2. Parse and execute the action
3. Append `Observation: <result>` to the scratchpad
4. Repeat until `finish(...)` or `MAX_ITERATIONS` reached

In [ ]:
REACT_SYSTEM = """You are a research assistant that answers questions by reasoning step by step
and using tools to retrieve evidence from a document corpus.

You MUST follow this exact format on every turn:
Thought: <your reasoning about what you know and what you still need>
Action: <exactly one of: search("query"), lookup("term"), finish("answer")>

Rules:
- Call search() for broad topic retrieval
- Call lookup() for a specific term or statistic you want to verify
- Call finish() ONLY when you have enough evidence to give a complete answer
- The answer inside finish() should be a full, cited response (cite [p.N] page numbers)
- Do NOT produce any text outside the Thought / Action block
- Do NOT call the same tool with the same argument twice"""

def rag_react(question: str,
              max_iterations: int = MAX_ITERATIONS,
              verbose: bool = True) -> Dict:
    t0 = time.time()
    scratchpad = f"Question: {question}\n"
    trace      = []   # list of {thought, action, tool, arg, observation}
    final_answer = None

    for iteration in range(1, max_iterations + 1):
        # Build prompt: system context + full scratchpad so far
        prompt = (
            f"{REACT_SYSTEM}\n\n"
            f"--- Scratchpad ---\n{scratchpad}\n"
            f"--- Continue ---\nThought:"
        )
        # LLM produces Thought + Action
        raw = llm(prompt).strip()
        # Ensure the response starts with 'Thought:' even if the model omits it
        if not raw.lower().startswith('thought:'):
            raw = 'Thought: ' + raw

        # Parse the action
        parsed = parse_action(raw)

        if parsed is None:
            # Model didn't emit a valid action — nudge it toward finish
            observation = "[System: no valid Action detected. Please emit Action: finish(\"answer\")]"
            tool, arg   = 'none', ''
        else:
            tool, arg = parsed
            if tool == 'finish':
                final_answer = arg
                trace.append({'iteration': iteration, 'thought': raw,
                              'tool': 'finish', 'arg': arg, 'observation': ''})
                if verbose:
                    print(f"  [{iteration}] FINISH")
                break
            elif tool in TOOL_REGISTRY:
                observation = TOOL_REGISTRY[tool](arg)
            else:
                observation = f"[Unknown tool '{tool}'. Use search, lookup, or finish.]"

        # Append to scratchpad
        scratchpad += f"Thought:{raw.split('Thought:',1)[-1]}\n"
        scratchpad += f"Observation: {observation[:600]}\n"

        # Extract thought text for trace
        thought_text = raw.split('Action:')[0].replace('Thought:', '').strip()
        trace.append({
            'iteration'  : iteration,
            'thought'    : thought_text,
            'tool'       : tool,
            'arg'        : arg,
            'observation': observation[:200],
        })

        if verbose:
            print(f"  [{iteration}] {tool}({arg[:50]!r})  "
                  f"→ {len(observation)} chars")

    # If max iterations hit without finish, force a synthesis
    if final_answer is None:
        synth_prompt = (
            f"{scratchpad}\n"
            f"You have reached the iteration limit. "
            f"Using everything in the scratchpad above, answer the original question "
            f"as completely as possible. Cite page numbers (e.g. [p.3]).\n\nAnswer:"
        )
        final_answer = llm(synth_prompt)
        if verbose:
            print(f"  [forced synthesis after {max_iterations} iterations]")

    latency = (time.time() - t0) * 1000

    if verbose:
        print(f"\n{'='*65}")
        print(f"Q: {question}")
        print(f"Iterations: {len(trace)}  |  Latency: {latency:.0f}ms")
        print(f"\nTrace:")
        for t in trace:
            print(f"  [{t['iteration']}] {t['tool']}({t['arg'][:50]!r})")
            print(f"       Thought: {t['thought'][:100]}")
            if t['observation']:
                print(f"       Obs    : {t['observation'][:100]}...")
        print(f"\nFinal answer:\n{final_answer}")
        print('-' * 65)

    return {
        'question'    : question,
        'trace'       : trace,
        'final_answer': final_answer,
        'iterations'  : len(trace),
        'latency_ms'  : latency,
    }

# Demo
rag_react("What are the feedback mechanisms that accelerate Arctic warming?")

## Step 9 — Comparison: Simple RAG vs ReAct RAG

For questions that require chained retrieval, ReAct's dynamic tool selection
should cover more pages and produce more grounded answers than a single-shot
simple RAG.

In [ ]:
def rag_simple(question: str, top_k: int = TOP_K) -> Dict:
    qvec    = embed_text(question)
    hits    = vs.search(qvec, top_k=top_k)
    context = '\n\n'.join(f"[p.{h['metadata'].get('page_num','?')}]\n{h['text']}" for h in hits)
    answer  = llm(
        f"Answer using ONLY the context. Cite page numbers.\n\n"
        f"Context:\n{context}\n\nQ: {question}\n\nA:"
    )
    pages = sorted({h['metadata'].get('page_num','?') for h in hits})
    return {'answer': answer, 'pages': pages}

test_questions = [
    "How do permafrost thaw and methane release interact with global warming?",
    "What evidence links greenhouse gas concentrations to observed temperature trends?",
    "How does ocean heat uptake moderate surface temperature rise?",
]

print(f"{'Question':<52}  {'Simple pages':>13}  {'ReAct pages':>12}  {'Iters':>6}")
print('-' * 92)

for q in test_questions:
    r_simple = rag_simple(q)
    r_react  = rag_react(q, verbose=False)

    react_pages = sorted({
        t['observation'][:6].split('.')[-1].strip(']')
        for t in r_react['trace']
        if t['observation']
    } - {''})

    print(f"{q[:51]:<52}  {str(r_simple['pages']):>13}  "
          f"{str(react_pages):>12}  {r_react['iterations']:>6}")

## Step 10 — Full ReAct Demo on a Complex Question

A question that requires the model to chase leads dynamically — the right follow-up
searches can't be known until after the first result is read.

In [ ]:
rag_react(
    "What are the projected consequences of a 2°C vs 4°C warming scenario "
    "for sea level, biodiversity, and extreme weather events?",
    max_iterations=8
)

## Step 11 — Summary

| Component | Implementation |
|-----------|---------------|
| ReAct format | Thought / Action / Observation in a text scratchpad |
| Tools | `search(query)` — semantic; `lookup(term)` — targeted; `finish(answer)` |
| Action parser | Regex on `Action: tool("arg")` format |
| Scratchpad | Full history passed to LLM on every iteration |
| Forced synthesis | If `MAX_ITERATIONS` hit without `finish`, one final synthesis call |
| Vector DB | Qdrant Cloud → OpenSearch → in-memory |
| Embeddings | Amazon Bedrock Titan V2 (1024-dim) |

## How ReAct compares to other Tier 3 patterns

| | CoT RAG (16) | **ReAct RAG (17)** |
|---|---|---|
| Plan determined | Upfront (fixed N steps) | Dynamically at each iteration |
| Each step informed by prior? | ✓ (prior findings in query) | ✓ (full scratchpad) |
| Can choose tool type? | ✗ (always vector search) | ✓ (`search` vs `lookup` vs `finish`) |
| Stops early if enough info? | ✗ (always runs all steps) | ✓ (`finish` on any iteration) |
| Failure mode | Over-planning simple Qs | Loops or misses `finish` format |

## When to use ReAct RAG

- The number of retrieval steps can't be known in advance
- Different question types warrant different retrieval strategies
- You want the model to self-terminate when it has enough evidence
- Questions with causal chains: *A leads to B, which causes C*

## Limitations

- **Format adherence**: model must reliably emit `Action: tool("arg")` — needs a capable LLM
- **Latency**: variable and potentially high (up to `MAX_ITERATIONS` LLM calls)
- **Scratchpad growth**: long histories inflate token count on every iteration

### Next: **18 — Memory-Augmented RAG**

In [ ]:
# vs.delete_collection()  # uncomment to clean up
print(f"Collection '{COLLECTION_NAME}' in {vs._backend} — {vs.count()} vectors")
print("\nDone. Give the go-ahead for notebook 18.")